# 7. Predictive Analysis — Classification
This notebook builds a reproducible landing-outcome classifier from labelled API records. It evaluates Logistic Regression, SVM, Decision Tree, and KNN. The dataset is filtered to rows with a known landing attempt and outcome. Because this live API snapshot can differ from the course data, report the source and row counts in your submission.

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "processed" / "spacex_falcon9_launches.csv")
df["landing_attempt"] = df["landing_attempt"].astype(str).str.lower().eq("true")
df["landing_success"] = pd.to_numeric(df["landing_success"], errors="coerce")
df = df[df["landing_attempt"] & df["landing_success"].notna()].copy()
df["Class"] = df["landing_success"].astype(int)
print("Labelled landing attempts:", len(df))
print(df["Class"].value_counts())
if df["Class"].nunique() < 2 or len(df) < 20:
    raise ValueError("Insufficient labelled examples or only one target class. Use the IBM course snapshot instead of live API data for the assignment.")
features = [c for c in ["flight_number","payload_mass_kg","launch_site","orbit","core_reused","core_flight","year"] if c in df.columns]
X, y = df[features], df["Class"]
num_cols = X.select_dtypes(include=["number","bool"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
numeric = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape, "Features:", features)

In [ ]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000), {"model__C": [0.1, 1, 10]}),
    "SVM": (SVC(), {"model__C": [0.1, 1, 10], "model__kernel": ["linear", "rbf"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42), {"model__max_depth": [3, 5, None], "model__min_samples_leaf": [1, 3, 5]}),
    "KNN": (KNeighborsClassifier(), {"model__n_neighbors": [3, 5, 7], "model__weights": ["uniform", "distance"]})
}
cv = StratifiedKFold(n_splits=min(5, int(y_train.value_counts().min())), shuffle=True, random_state=42)
results, fitted = [], {}
for name, (estimator, params) in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    search = GridSearchCV(pipe, params, cv=cv, scoring="f1", n_jobs=-1)
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    row = {
        "model": name,
        "best_params": str(search.best_params_),
        "cv_best_f1": search.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred, zero_division=0),
        "test_recall": recall_score(y_test, pred, zero_division=0),
        "test_f1": f1_score(y_test, pred, zero_division=0)
    }
    results.append(row); fitted[name] = (search.best_estimator_, pred)
metrics = pd.DataFrame(results).sort_values(["cv_best_f1","test_f1"], ascending=False)
display(metrics)
out = ROOT / "outputs" / "model_results"
out.mkdir(parents=True, exist_ok=True)
metrics.to_csv(out / "model_comparison.csv", index=False)
best_name = metrics.iloc[0]["model"]
best_model, best_pred = fitted[best_name]
print("Selected model by cross-validated F1:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, best_pred, labels=[0,1])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted fail","Predicted success"],
            yticklabels=["Actual fail","Actual success"])
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "figures" / "best_model_confusion_matrix.png", dpi=160)
plt.show()
print("Confusion matrix [actual rows, predicted columns]:\n", cm)

## Interpretation and limitations
Use the metrics printed above in the presentation. Explain precision, recall, F1-score, and the four confusion-matrix cells. The selected model is chosen by cross-validated F1 in this notebook, with test metrics reported separately. This is an educational historical analysis, not a mission-safety system. Check for temporal leakage and compare against the official IBM course dataset before final submission.